# 🔋 Battery State-of-Health Estimation: Physics-Informed Deep Learning

**WIDSS — Windowed Intelligent Drive-cycle State eStimation**

## Project Overview
This notebook demonstrates an end-to-end pipeline for battery health monitoring using:
- **Physics-based feature engineering** (Single Particle Model)
- **Realistic aging simulation** with capacity fade & resistance growth
- **Transformer-based architectures** for temporal sequence modeling
- **Comparative analysis** with LSTM baselines
- **Predictive degradation modeling** for remaining useful life (RUL) estimation

## Key Innovation
Rather than treating voltage & current as black-box inputs, we extract physics-informed features (lithium stoichiometry, overpotential) that directly relate to electrochemical behavior. This hybrid physics-ML approach enables interpretable, data-efficient degradation monitoring.

## 0. Environment Setup & Imports

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from widss.simulation import BatterySimulationConfig, build_dataset
from widss.dataset import build_sequences
from widss.evaluation import rmse, mae, mape

# ════════════════════════════════════════════════════════════════
# DARK MODE STYLING (Black Background + White Text)
# ════════════════════════════════════════════════════════════════
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': '#1a1a1a',
    'axes.facecolor': '#0d0d0d',
    'axes.edgecolor': '#404040',
    'axes.grid': True,
    'grid.alpha': 0.2,
    'grid.color': '#505050',
    'axes.labelcolor': '#ffffff',
    'text.color': '#ffffff',
    'xtick.color': '#ffffff',
    'ytick.color': '#ffffff',
    'font.size': 10,
    'legend.fontsize': 9,
    'legend.facecolor': '#1a1a1a',
    'legend.edgecolor': '#404040'
})
sns.set_palette('husl')

SEED = 42
rng = np.random.default_rng(SEED)
print('✅ Environment ready for battery SoH estimation (Dark Mode enabled)')

---
## Part 1: Physics-Based Feature Extraction with SPM

### Background
The **Single Particle Model (SPM)** is a reduced-order electrochemical model that captures the essential physics of Li-ion cells:
- **Lithium diffusion** in spherical particles (anode & cathode)
- **Butler-Volmer kinetics** for charge transfer reactions
- **Open-circuit potential (OCP)** as a function of stoichiometry

### Why SPM Features Matter
1. **Interpretability**: θ_neg, θ_pos directly correspond to lithium inventory
2. **Physics grounding**: Overpotential η relates to reaction kinetics and aging mechanisms
3. **Early fault detection**: Concentration gradients hint at plating or degradation

### Implementation
We use a **first-order ODE approximation** of diffusion, suitable for real-time computation on resource-constrained systems.

In [ ]:
class SimpleSPM:
    """
    Single Particle Model (SPM) — Reduced-Order Electrochemical Model
    
    This implementation models lithium transport in a spherical particle framework,
    capturing the relationship between current input and electrochemical state.
    
    Mathematical Foundation:
    ├─ Solid-phase diffusion (Fick's law, 1D spherical)
    ├─ Surface concentration dynamics (mass balance at electrode/electrolyte interface)
    ├─ Open-circuit potential (polynomial/exponential fits from measurements)
    └─ Butler-Volmer overpotential (current-voltage relationship)
    
    References:
    - Moura et al., IEEE Transactions on Control Systems Technology, vol. 25, no. 3, 2017
    - Perez et al., Journal of Power Sources, vol. 196, no. 1, pp. 430-441, 2011
    """

    def __init__(self,
                 # Negative Electrode (Graphite Anode) Parameters
                 Ds_neg: float = 3.9e-14,       # Solid diffusivity [m²/s]
                 R_neg: float = 12.5e-6,        # Particle radius [m]
                 eps_neg: float = 0.3,          # Volume fraction [dimensionless]
                 L_neg: float = 100e-6,         # Electrode thickness [m]
                 
                 # Positive Electrode (NCA/NMC Cathode) Parameters
                 Ds_pos: float = 1.0e-13,       # Solid diffusivity [m²/s]
                 R_pos: float = 10.0e-6,        # Particle radius [m]
                 eps_pos: float = 0.3,          # Volume fraction [dimensionless]
                 L_pos: float = 80e-6,          # Electrode thickness [m]
                 
                 # Cell & Material Properties
                 F: float = 96485.0,            # Faraday constant [C/mol]
                 A: float = 1.0,                # Electrode area [m²]
                 c_s_neg_max: float = 30555.0,  # Max Li concentration anode [mol/m³]
                 c_s_pos_max: float = 51554.0,  # Max Li concentration cathode [mol/m³]
                 soc_init: float = 0.95,        # Initial SoC [0-1]
                 dt: float = 1.0)               # Time discretization [s]

        # Store parameters
        self.F, self.A, self.dt = F, A, dt
        self.Ds_neg, self.R_neg, self.eps_neg, self.L_neg = Ds_neg, R_neg, eps_neg, L_neg
        self.Ds_pos, self.R_pos, self.eps_pos, self.L_pos = Ds_pos, R_pos, eps_pos, L_pos
        self.c_s_neg_max = c_s_neg_max
        self.c_s_pos_max = c_s_pos_max

        # Initialize lithium concentrations from SoC
        theta_neg_0 = 0.8 + soc_init * (0.8 - 0.8)  # Simplified stoichiometry mapping
        theta_pos_0 = 1.0 - soc_init * 0.5
        self.c_neg = theta_neg_0 * c_s_neg_max
        self.c_pos = theta_pos_0 * c_s_pos_max

        # Characteristic diffusion time constants [s]
        self.tau_neg = self.R_neg**2 / (15 * self.Ds_neg)
        self.tau_pos = self.R_pos**2 / (15 * self.Ds_pos)

        # Interfacial area density [m²/m³]
        self.a_neg = 3 * eps_neg / R_neg
        self.a_pos = 3 * eps_pos / R_pos

    def step(self, current_a: float) -> dict:
        """
        Advance SPM by one timestep.
        
        Args:
            current_a: Current [A] (positive = discharging, negative = charging)
        
        Returns:
            Dictionary with computed states and features
        """
        I = current_a

        # ─ Molar flux at electrode/electrolyte interface ────────────────────
        # j = I / (F * a * L * A)  [mol/m² s]
        j_neg = I / (self.F * self.a_neg * self.L_neg * self.A)
        j_pos = -I / (self.F * self.a_pos * self.L_pos * self.A)

        # ─ Surface concentration dynamics (first-order approximation) ───────
        # dc_s/dt = -3*j/R (conservation of lithium at surface)
        dc_neg = -3 * j_neg / self.R_neg
        dc_pos = -3 * j_pos / self.R_pos
        self.c_neg += dc_neg * self.dt
        self.c_pos += dc_pos * self.dt

        # Physical bounds enforcement
        self.c_neg = np.clip(self.c_neg, 0, self.c_s_neg_max)
        self.c_pos = np.clip(self.c_pos, 0, self.c_s_pos_max)

        # Normalize to stoichiometry [dimensionless, 0-1]
        theta_neg = self.c_neg / self.c_s_neg_max
        theta_pos = self.c_pos / self.c_s_pos_max

        # ─ Open-Circuit Potential (OCP) ─────────────────────────────────────
        # Fitted polynomial/exponential models from measurement data
        ocp_neg = (0.194 + 1.5 * np.exp(-120 * theta_neg) - 
                   0.045 * np.tanh(15 * (theta_neg - 0.42)))  # [V vs Li/Li+]
        
        ocp_pos = (4.19829 + 0.0565661 * np.tanh(-14.5546 * theta_pos + 8.60) - 
                   0.0275479 * (1 / (0.998432 - theta_pos)**0.492465 - 1.90111) - 
                   0.157123 * np.exp(-0.04738 * theta_pos**8))  # [V vs Li/Li+]

        # ─ Butler-Volmer Overpotential ───────────────────────────────────────
        # η = (RT/F) * sinh⁻¹(j / j0)  ≈ j / (F*k0*√[θ(1-θ)])
        # where j0 is the exchange current density
        eta_neg = j_neg / (1e-4 * np.sqrt(theta_neg * (1 - theta_neg) + 1e-6))
        eta_pos = j_pos / (1e-4 * np.sqrt(theta_pos * (1 - theta_pos) + 1e-6))

        return {
            'theta_neg': theta_neg,
            'theta_pos': theta_pos,
            'ocp_neg': ocp_neg,
            'ocp_pos': ocp_pos,
            'eta_neg': eta_neg,
            'eta_pos': eta_pos,
            'j_neg': j_neg,
            'j_pos': j_pos
        }

print('✅ SimpleSPM class defined with full documentation')

In [ ]:
# ── Dataset Generation with SPM Feature Extraction ────────────────────────────
print('Building synthetic drive-cycle dataset...')
cfg = BatterySimulationConfig(capacity_ah=60.0, soc_init=0.95, dt_s=1.0)
df_base = build_dataset(duration_s=7200, config=cfg, seed=SEED)

print(f'Base dataset shape: {df_base.shape}')
print(f'Base features: {list(df_base.columns)}')
print(f'Current range: [{df_base.current_a.min():.2f}, {df_base.current_a.max():.2f}] A')
print(f'Voltage range: [{df_base.voltage_v.min():.2f}, {df_base.voltage_v.max():.2f}] V')
print(f'SoC range: [{df_base.soc.min():.3f}, {df_base.soc.max():.3f}]')

# Initialize SPM and compute physics features
spm = SimpleSPM(soc_init=0.95, dt=1.0)
spm_records = [spm.step(I) for I in df_base['current_a'].values]
df_spm = pd.DataFrame(spm_records)

# Combine base and SPM-derived features
df_phys = pd.concat([df_base.reset_index(drop=True), df_spm], axis=1)
print(f'\n✅ Enhanced dataset created: {df_phys.shape}')
print(f'SPM features added: {list(df_spm.columns)}')

In [ ]:
# ── Visualization: SPM Features vs SoC ──────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
axes = axes.flatten()

spm_cols = ['theta_neg', 'theta_pos', 'ocp_neg', 'ocp_pos', 'eta_neg', 'eta_pos']
titles = [
    'Anode Stoichiometry (θ_neg)',
    'Cathode Stoichiometry (θ_pos)',
    'Anode OCP (V)',
    'Cathode OCP (V)',
    'Anode Overpotential (V)',
    'Cathode Overpotential (V)'
]
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E', '#BC4749']

for ax, col, title, color in zip(axes, spm_cols, titles, colors):
    scatter = ax.scatter(df_phys['soc'], df_phys[col], s=2, alpha=0.4, color=color)
    ax.set_xlabel('State of Charge (SoC)', fontsize=10, color='white')
    ax.set_ylabel(col, fontsize=10, color='white')
    ax.set_title(title, fontsize=11, fontweight='bold', color='white')
    ax.grid(True, alpha=0.2, color='#505050')
    ax.tick_params(colors='white')

plt.suptitle('Physics-Informed Features: SPM Output Landscape', 
             fontsize=14, fontweight='bold', y=0.995, color='white')
plt.tight_layout()
plt.show()

print('\n📊 Observations:')
print('  • θ_neg & θ_pos show strong monotonic relationship with SoC')
print('  • OCP curves are highly nonlinear (characteristic of Li-ion)')
print('  • Overpotential spikes during high-current transients')

---
## Part 2: Battery Aging Simulation & SoH Modeling

### Aging Mechanisms
Real batteries degrade through multiple mechanisms:
1. **SEI (Solid Electrolyte Interphase) Growth** → Capacity fade
2. **Structural Changes** → Loss of active material
3. **Electrolyte Decomposition** → Increased impedance
4. **Lithium Plating** → Safety hazard and irreversible capacity loss

### Model Formulation
We implement an **empirical degradation model** fitted to real-world aging data patterns:
- Capacity: Q(n) = Q₀ × exp(-α×n)
- Resistance: R(n) = R₀ × (1 + β×√n)

### End-of-Life Criterion
Battery reaches end-of-life when SoH drops to **80%** (industry standard for plug-in hybrids)

In [ ]:
def simulate_aging(n_cycles: int = 500,
                   capacity_init_ah: float = 60.0,
                   resistance_init_ohm: float = 0.02,
                   alpha: float = 0.00015,
                   beta: float = 0.0001,
                   noise_std: float = 0.001,
                   seed: int = 42) -> pd.DataFrame:
    """
    Simulate battery aging over multiple charge-discharge cycles.
    
    Model:
        Capacity(n) = Q0 * exp(-α*n) + noise
        Resistance(n) = R0 * (1 + β*√n) + noise
        SoH(n) = Capacity(n) / Q0
    
    Args:
        n_cycles: Number of cycles to simulate
        capacity_init_ah: Initial usable capacity [Ah]
        resistance_init_ohm: Initial internal resistance [Ω]
        alpha: Exponential fade rate [1/cycle]
        beta: Resistance growth rate [1/√cycle]
        noise_std: Measurement noise standard deviation
        seed: Random seed for reproducibility
    
    Returns:
        DataFrame with cycle-by-cycle degradation data
    """
    rng = np.random.default_rng(seed)
    cycles = np.arange(1, n_cycles + 1)

    # Exponential capacity fade (characteristic of SEI growth)
    capacity = (capacity_init_ah * np.exp(-alpha * cycles) + 
                rng.normal(0, noise_std, n_cycles))

    # Square-root resistance growth (diffusional impedance increase)
    resistance = (resistance_init_ohm * (1 + beta * np.sqrt(cycles)) + 
                  rng.normal(0, noise_std * 0.001, n_cycles))

    # State of Health: relative capacity
    soh = capacity / capacity_init_ah

    return pd.DataFrame({
        'cycle': cycles,
        'capacity_ah': capacity,
        'resistance_ohm': resistance,
        'soh': soh
    })

# Generate aging trajectory
df_aging = simulate_aging(n_cycles=1000, alpha=0.00015, beta=0.0001)
print(f'✅ Aging simulation: {df_aging.shape[0]} cycles')
print(f'SoH trajectory: {df_aging.soh.max():.3f} → {df_aging.soh.min():.3f}')
print(f'Resistance growth: {df_aging.resistance_ohm.iloc[0]*1000:.3f} mΩ → {df_aging.resistance_ohm.iloc[-1]*1000:.3f} mΩ')

In [ ]:
# ── Degradation Curves Visualization ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Capacity fade
axes[0].fill_between(df_aging['cycle'], df_aging['capacity_ah'], alpha=0.3, color='#2E86AB')
axes[0].plot(df_aging['cycle'], df_aging['capacity_ah'], color='#2E86AB', lw=2, label='Simulated capacity')
axes[0].axhline(0.8 * 60, color='#C73E1D', ls='--', lw=2, label='80% threshold (EoL)')
axes[0].set_title('Capacity Fade Over Cycles', fontsize=12, fontweight='bold', color='white')
axes[0].set_xlabel('Cycle Number', color='white'); axes[0].set_ylabel('Capacity (Ah)', color='white')
axes[0].legend(loc='upper right'); axes[0].grid(True, alpha=0.2, color='#505050')
axes[0].tick_params(colors='white')

# Resistance growth
axes[1].fill_between(df_aging['cycle'], df_aging['resistance_ohm']*1000, alpha=0.3, color='#F18F01')
axes[1].plot(df_aging['cycle'], df_aging['resistance_ohm']*1000, color='#F18F01', lw=2, label='Internal resistance')
axes[1].set_title('Impedance Growth', fontsize=12, fontweight='bold', color='white')
axes[1].set_xlabel('Cycle Number', color='white'); axes[1].set_ylabel('Resistance (mΩ)', color='white')
axes[1].grid(True, alpha=0.2, color='#505050')
axes[1].tick_params(colors='white')

# SoH with EoL marker
axes[2].fill_between(df_aging['cycle'], df_aging['soh']*100, alpha=0.3, color='#6A994E')
axes[2].plot(df_aging['cycle'], df_aging['soh']*100, color='#6A994E', lw=2, label='State of Health')
axes[2].axhline(80, color='#C73E1D', ls='--', lw=2, label='80% SoH threshold')

# Find and mark EoL
eol_cycle = df_aging[df_aging['soh'] <= 0.8]['cycle'].min()
axes[2].axvline(eol_cycle, color='#BC4749', ls=':', lw=2.5, label=f'EoL @ cycle {int(eol_cycle)}')
axes[2].scatter([eol_cycle], [80], color='#BC4749', s=150, marker='X', zorder=5)

axes[2].set_title('State of Health (SoH) Trajectory', fontsize=12, fontweight='bold', color='white')
axes[2].set_xlabel('Cycle Number', color='white'); axes[2].set_ylabel('SoH (%)', color='white')
axes[2].set_ylim([75, 105])
axes[2].legend(loc='upper right'); axes[2].grid(True, alpha=0.2, color='#505050')
axes[2].tick_params(colors='white')

plt.suptitle('Battery Aging Simulation: Three-View Analysis', fontsize=14, fontweight='bold', color='white')
plt.tight_layout()
plt.show()

print(f'\n🎯 End-of-Life Analysis:')
print(f'  Cycle at EoL (SoH≤80%): {int(eol_cycle)}')
print(f'  Expected service life: ~{int(eol_cycle)} charge cycles')
print(f'  Capacity fade rate: {df_aging.iloc[:100].soh.mean() - df_aging.iloc[-100:].soh.mean():.4f} per 100 cycles (avg)')

---
## Part 3: SoH Prediction Using Gradient Boosting

### Baseline Model
Before deploying deep learning, we establish a **tree-based baseline** for comparison.
Gradient Boosting Regression captures non-linear aging patterns effectively.

### Features
- Cycle number (temporal trend)
- Measured resistance (direct aging indicator)

### Evaluation Metrics
- **RMSE**: Overall prediction accuracy
- **MAE**: Median absolute error (robust to outliers)


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

# Prepare data for SoH regression
X_age = df_aging[['cycle', 'resistance_ohm']].values
y_age = df_aging['soh'].values

# Train-test split (80-20)
split_age = int(0.8 * len(df_aging))
X_tr_a, X_te_a = X_age[:split_age], X_age[split_age:]
y_tr_a, y_te_a = y_age[:split_age], y_age[split_age:]

# Standardize features
sc_a = StandardScaler()

# Train Gradient Boosting model
print('Training Gradient Boosting SoH predictor...')
gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    min_samples_split=5,
    random_state=SEED,
    verbose=0
)
gbr.fit(sc_a.fit_transform(X_tr_a), y_tr_a)
y_soh_pred = gbr.predict(sc_a.transform(X_te_a))

# Evaluation
gbr_rmse = rmse(y_te_a, y_soh_pred)
gbr_mae = mae(y_te_a, y_soh_pred)

print(f'\n📊 Gradient Boosting Results (Test Set):')  
print(f'  RMSE: {gbr_rmse:.5f}')
print(f'  MAE:  {gbr_mae:.5f}')
print(f'  Max Error: {np.abs(y_te_a - y_soh_pred).max():.5f}')

In [ ]:
# ── Baseline Model Visualization ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: Full trajectory
test_cycles = df_aging['cycle'].values[split_age:]
axes[0].plot(test_cycles, y_te_a, 'o-', color='#2E86AB', lw=2, markersize=3, 
             label='Ground Truth', alpha=0.8)
axes[0].plot(test_cycles, y_soh_pred, 's--', color='#F18F01', lw=2, markersize=3,
             label='GBR Prediction', alpha=0.8)
axes[0].fill_between(test_cycles, y_te_a - gbr_mae, y_te_a + gbr_mae, 
                      color='gray', alpha=0.2, label='±MAE band')
axes[0].set_xlabel('Cycle Number', fontsize=11, color='white')
axes[0].set_ylabel('State of Health', fontsize=11, color='white')
axes[0].set_title('SoH Prediction: Gradient Boosting Baseline', fontsize=12, fontweight='bold', color='white')
axes[0].legend(); axes[0].grid(True, alpha=0.2, color='#505050')
axes[0].tick_params(colors='white')

# Right: Residuals
residuals = y_te_a - y_soh_pred
axes[1].scatter(test_cycles, residuals*100, s=20, alpha=0.6, color='#A23B72')
axes[1].axhline(0, color='#ffffff', ls='-', lw=0.8)
axes[1].fill_between(test_cycles, -gbr_mae*100, gbr_mae*100, 
                      color='red', alpha=0.1, label='±MAE')
axes[1].set_xlabel('Cycle Number', fontsize=11, color='white')
axes[1].set_ylabel('Prediction Error (% SoH)', fontsize=11, color='white')
axes[1].set_title('Residual Analysis', fontsize=12, fontweight='bold', color='white')
axes[1].legend(); axes[1].grid(True, alpha=0.2, color='#505050')
axes[1].tick_params(colors='white')

plt.tight_layout()
plt.show()

---
## Part 4: Transformer Architecture for SoC Estimation

### Why Transformers?
**Self-Attention Mechanism** allows the model to:
1. Learn which timesteps are most relevant for current SoC prediction
2. Capture long-range dependencies (e.g., hour-long charging patterns)
3. Parallelize computation (vs. RNN sequential bottleneck)

### Architecture
```
Input (60 timesteps, 6 SPM features)
    ↓
Linear Projection → 32 dims
    ↓
Positional Encoding (learnable)
    ↓
[Multi-Head Attention × 2] + LayerNorm + FFN
    ↓
Global Average Pooling
    ↓
Fully Connected (32 → 1) with Sigmoid
    ↓
SoC ∈ [0, 1]
```


In [ ]:
from widss.model import tensorflow_available

WINDOW_SIZE = 60  # 60-second temporal window

if not tensorflow_available():
    print('⚠️  TensorFlow not installed — Transformer section will be skipped.')
    print('    Install via: pip install tensorflow')
else:
    import tensorflow as tf
    from tensorflow.keras import layers, Model
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

    # ─────────────────────────────────────────────────────────────────────────
    class TransformerBlock(layers.Layer):
        """Transformer Encoder Block: Multi-Head Attention + Position-wise FFN.
        
        Architecture:
            Input → [MultiHeadAttention] ─┐
                         ↓               ├─→ Add & Norm ─┐
                      [Dropout]         │               │
                                        │          [FFN (Dense×2)] ─┐
                                        │               ↓           ├→ Add & Norm → Output
                                        │            [Dropout]      │
                                        └─────────────────────────┘
        """

        def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, 
                     dropout_rate: float = 0.1, **kwargs):
            super().__init__(**kwargs)
            
            # Multi-head self-attention
            self.att = layers.MultiHeadAttention(
                num_heads=num_heads, 
                key_dim=embed_dim // num_heads,
                dropout=dropout_rate
            )
            
            # Feed-forward network (position-wise)
            self.ffn = tf.keras.Sequential([
                layers.Dense(ff_dim, activation='relu'),
                layers.Dense(embed_dim)
            ])
            
            # Layer normalization (post-attention & post-FFN)
            self.ln1 = layers.LayerNormalization(epsilon=1e-6)
            self.ln2 = layers.LayerNormalization(epsilon=1e-6)
            
            # Dropout for regularization
            self.drop1 = layers.Dropout(dropout_rate)
            self.drop2 = layers.Dropout(dropout_rate)

        def call(self, x, training=False):
            # Self-attention branch
            attn_output = self.att(x, x, training=training)
            attn_output = self.drop1(attn_output, training=training)
            out1 = self.ln1(x + attn_output)  # Residual + LayerNorm
            
            # Feed-forward branch
            ffn_output = self.ffn(out1)
            ffn_output = self.drop2(ffn_output, training=training)
            out2 = self.ln2(out1 + ffn_output)  # Residual + LayerNorm
            
            return out2

    # ─────────────────────────────────────────────────────────────────────────
    def build_transformer_soc(
        window_size: int,
        feature_count: int,
        embed_dim: int = 32,
        num_heads: int = 4,
        ff_dim: int = 64,
        num_blocks: int = 2,
        dropout: float = 0.1,
        lr: float = 1e-3
    ):
        """Build a Transformer-based SoC estimator.
        
        Args:
            window_size: Temporal context length (timesteps)
            feature_count: Number of input features per timestep
            embed_dim: Embedding dimension (transformer dimension)
            num_heads: Number of attention heads
            ff_dim: Feed-forward hidden dimension
            num_blocks: Number of stacked transformer blocks
            dropout: Dropout rate
            lr: Learning rate for Adam optimizer
        
        Returns:
            Compiled Keras model
        """
        # Input layer
        inp = layers.Input(shape=(window_size, feature_count), name='spm_features')

        # Project raw features to embedding dimension
        x = layers.Dense(embed_dim, name='input_projection')(inp)

        # Learnable positional embeddings (absolute position encoding)
        positions = tf.range(start=0, limit=window_size, delta=1)
        pos_emb = layers.Embedding(
            input_dim=window_size, 
            output_dim=embed_dim,
            name='positional_embedding'
        )(positions)
        x = x + pos_emb  # Broadcast addition across batch

        # Stack Transformer encoder blocks
        for i in range(num_blocks):
            x = TransformerBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ff_dim=ff_dim,
                dropout_rate=dropout,
                name=f'transformer_block_{i+1}'
            )(x, training=True)

        # Global temporal pooling (average over time dimension)
        x = layers.GlobalAveragePooling1D(name='temporal_pooling')(x)

        # Dense classifier head
        x = layers.Dense(32, activation='relu', name='dense_1')(x)
        x = layers.Dropout(dropout, name='dropout')(x)
        out = layers.Dense(1, activation='sigmoid', name='soc_output')(x)  # SoC ∈ [0,1]

        model = Model(inp, out, name='TransformerSoC')
        
        # Compile with MSE loss (regression task)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
            loss='mse',
            metrics=['mae']
        )
        return model

    print('✅ Transformer architecture implemented')

In [ ]:
if tensorflow_available():
    # Feature list for sequence construction
    SPM_FEAT_COLS = ['voltage_v', 'current_a', 'theta_neg', 'theta_pos', 'eta_neg', 'eta_pos']

    def build_spm_sequences(df_full, feature_cols, window_size=60):
        """Create sliding window sequences from time series data.
        
        Args:
            df_full: DataFrame with all features
            feature_cols: List of column names to use
            window_size: Number of timesteps per sequence
        
        Returns:
            X: Array of shape (n_samples, window_size, n_features)
            y: Target SoC values
        """
        data = df_full[feature_cols].values.astype('float32')
        target = df_full['soc'].values.astype('float32')
        
        # Sliding window
        X = np.stack(
            [data[i:i+window_size] for i in range(len(data) - window_size)],
            axis=0
        )
        y = target[window_size:]  # Target is the SoC at the end of each window
        return X, y

    # Normalize features to zero-mean, unit-variance
    from sklearn.preprocessing import StandardScaler as SC
    sc_spm = SC()
    df_phys_norm = df_phys.copy()
    df_phys_norm[SPM_FEAT_COLS] = sc_spm.fit_transform(df_phys[SPM_FEAT_COLS])

    # Build sequences
    X_spm, y_spm = build_spm_sequences(df_phys_norm, SPM_FEAT_COLS, WINDOW_SIZE)
    
    # Train-test split
    split_t = int(0.8 * len(X_spm))
    X_tr_t, X_te_t = X_spm[:split_t], X_spm[split_t:]
    y_tr_t, y_te_t = y_spm[:split_t], y_spm[split_t:]
    
    print(f'✅ SPM sequence dataset prepared')
    print(f'  Training set: {X_tr_t.shape}')
    print(f'  Test set: {X_te_t.shape}')
    print(f'  Features: {SPM_FEAT_COLS}')

In [ ]:
if tensorflow_available():
    # Build Transformer model
    print('Building Transformer-based SoC estimator...')
    tf.random.set_seed(SEED)
    np.random.seed(SEED)
    
    transformer = build_transformer_soc(
        window_size=WINDOW_SIZE,
        feature_count=len(SPM_FEAT_COLS),
        embed_dim=32,
        num_heads=4,
        ff_dim=64,
        num_blocks=2,
        dropout=0.1,
        lr=1e-3
    )
    
    print('\nModel Architecture:')
    transformer.summary()

In [ ]:
if tensorflow_available():
    # Training callbacks
    callbacks_t = [
        EarlyStopping(
            monitor='val_loss',
            patience=6,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        )
    ]

    print('Training Transformer model...')
    history_t = transformer.fit(
        X_tr_t, y_tr_t,
        validation_split=0.15,
        epochs=30,
        batch_size=64,
        callbacks=callbacks_t,
        verbose=1
    )

In [ ]:
if tensorflow_available():
    # Generate predictions on test set
    y_trans_pred = np.clip(transformer.predict(X_te_t, verbose=0).flatten(), 0, 1)
    
    trans_rmse = rmse(y_te_t, y_trans_pred)
    trans_mae = mae(y_te_t, y_trans_pred)
    trans_mape = mape(y_te_t, y_trans_pred)
    
    print('\n' + '='*60)
    print('📊 TRANSFORMER RESULTS (Test Set with SPM Features)')
    print('='*60)
    print(f'  RMSE : {trans_rmse:.5f}')
    print(f'  MAE  : {trans_mae:.5f}')
    print(f'  MAPE : {trans_mape:.2f}%')
    print(f'  Max Error: {np.abs(y_te_t - y_trans_pred).max():.5f}')
    print('='*60)

---
## Part 5: LSTM Baseline vs Transformer Comparison

In [ ]:
if tensorflow_available():
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import LSTM, Dropout, Dense

    print('Building LSTM baseline...')
    
    lstm_model = Sequential([
        LSTM(64, return_sequences=True, 
             input_shape=(WINDOW_SIZE, len(SPM_FEAT_COLS)),
             name='lstm_1'),
        Dropout(0.2, name='dropout_1'),
        LSTM(32, name='lstm_2'),
        Dropout(0.2, name='dropout_2'),
        Dense(32, activation='relu', name='dense_1'),
        Dropout(0.2, name='dropout_3'),
        Dense(1, activation='sigmoid', name='soc_output')
    ], name='LSTM_SoC')
    
    lstm_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='mse',
        metrics=['mae']
    )

    print('Training LSTM model...')
    history_lstm = lstm_model.fit(
        X_tr_t, y_tr_t,
        validation_split=0.15,
        epochs=30,
        batch_size=64,
        callbacks=callbacks_t,
        verbose=0
    )

    y_lstm_pred = np.clip(lstm_model.predict(X_te_t, verbose=0).flatten(), 0, 1)
    
    lstm_rmse = rmse(y_te_t, y_lstm_pred)
    lstm_mae = mae(y_te_t, y_lstm_pred)
    lstm_mape = mape(y_te_t, y_lstm_pred)
    
    print('\n' + '='*60)
    print('📊 LSTM RESULTS (Test Set with SPM Features)')
    print('='*60)
    print(f'  RMSE : {lstm_rmse:.5f}')
    print(f'  MAE  : {lstm_mae:.5f}')
    print(f'  MAPE : {lstm_mape:.2f}%')
    print(f'  Max Error: {np.abs(y_te_t - y_lstm_pred).max():.5f}')
    print('='*60)

In [ ]:
if tensorflow_available():
    # Comparative results table
    comparison_df = pd.DataFrame({
        'Model': ['LSTM (2-layer)', 'Transformer (2-block)'],
        'RMSE': [lstm_rmse, trans_rmse],
        'MAE': [lstm_mae, trans_mae],
        'MAPE (%)': [lstm_mape, trans_mape],
        'Max Error': [np.abs(y_te_t - y_lstm_pred).max(),
                      np.abs(y_te_t - y_trans_pred).max()]
    })
    
    print('\n🏆 MODEL COMPARISON TABLE')
    print(comparison_df.to_string(index=False))
    
    # Determine winner
    if lstm_rmse < trans_rmse:
        winner = 'LSTM'
        improvement = (1 - trans_rmse/lstm_rmse) * 100
    else:
        winner = 'Transformer'
        improvement = (1 - lstm_rmse/trans_rmse) * 100
    
    print(f'\n🎯 Winner: {winner} ({improvement:.1f}% RMSE improvement)')

In [ ]:
if tensorflow_available():
    # Detailed prediction comparison
    fig, axes = plt.subplots(2, 2, figsize=(16, 9))
    
    plot_range = min(800, len(y_te_t))  # Plot first 800 test samples
    steps = np.arange(plot_range)
    
    # ─ LSTM Predictions ───────────────────────────────────────────────────
    axes[0, 0].plot(steps, y_te_t[:plot_range], 'o-', color='#00ff00', lw=2, markersize=3, label='Ground Truth', alpha=0.8)
    axes[0, 0].plot(steps, y_lstm_pred[:plot_range], 's--', color='#00ccff', lw=1.5, label='LSTM', alpha=0.7)
    axes[0, 0].fill_between(steps, y_te_t[:plot_range] - lstm_mae, 
                            y_te_t[:plot_range] + lstm_mae,
                            color='#0099ff', alpha=0.1)
    axes[0, 0].set_title(f'LSTM — RMSE: {lstm_rmse:.4f}', fontsize=12, fontweight='bold', color='white')
    axes[0, 0].set_ylabel('SoC', fontsize=11, color='white'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.2, color='#505050')
    axes[0, 0].tick_params(colors='white')
    
    # ─ Transformer Predictions ────────────────────────────────────────────
    axes[0, 1].plot(steps, y_te_t[:plot_range], 'o-', color='#00ff00', lw=2, markersize=3, label='Ground Truth', alpha=0.8)
    axes[0, 1].plot(steps, y_trans_pred[:plot_range], 's--', color='#ff6600', lw=1.5, label='Transformer', alpha=0.7)
    axes[0, 1].fill_between(steps, y_te_t[:plot_range] - trans_mae,
                            y_te_t[:plot_range] + trans_mae,
                            color='#ff6600', alpha=0.1)
    axes[0, 1].set_title(f'Transformer — RMSE: {trans_rmse:.4f}', fontsize=12, fontweight='bold', color='white')
    axes[0, 1].set_ylabel('SoC', fontsize=11, color='white'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.2, color='#505050')
    axes[0, 1].tick_params(colors='white')
    
    # ─ LSTM Residuals ─────────────────────────────────────────────────────
    lstm_residuals = (y_te_t[:plot_range] - y_lstm_pred[:plot_range]) * 100
    axes[1, 0].scatter(steps, lstm_residuals, s=10, alpha=0.5, color='#00ccff')
    axes[1, 0].axhline(0, color='#ffffff', ls='-', lw=0.8)
    axes[1, 0].fill_between(steps, -lstm_mae*100, lstm_mae*100, color='#0099ff', alpha=0.1)
    axes[1, 0].set_xlabel('Timestep', fontsize=11, color='white'); axes[1, 0].set_ylabel('Error (%)', fontsize=11, color='white')
    axes[1, 0].set_title('LSTM Residuals', fontsize=12, fontweight='bold', color='white'); axes[1, 0].grid(True, alpha=0.2, color='#505050')
    axes[1, 0].tick_params(colors='white')
    
    # ─ Transformer Residuals ──────────────────────────────────────────────
    trans_residuals = (y_te_t[:plot_range] - y_trans_pred[:plot_range]) * 100
    axes[1, 1].scatter(steps, trans_residuals, s=10, alpha=0.5, color='#ff6600')
    axes[1, 1].axhline(0, color='#ffffff', ls='-', lw=0.8)
    axes[1, 1].fill_between(steps, -trans_mae*100, trans_mae*100, color='#ff6600', alpha=0.1)
    axes[1, 1].set_xlabel('Timestep', fontsize=11, color='white'); axes[1, 1].set_ylabel('Error (%)', fontsize=11, color='white')
    axes[1, 1].set_title('Transformer Residuals', fontsize=12, fontweight='bold', color='white'); axes[1, 1].grid(True, alpha=0.2, color='#505050')
    axes[1, 1].tick_params(colors='white')
    
    plt.suptitle('LSTM vs Transformer: Prediction Analysis (first 800 test steps)',
                 fontsize=14, fontweight='bold', color='white')
    plt.tight_layout()
    plt.show()

In [ ]:
if tensorflow_available():
    # Training dynamics comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    epochs_lstm = range(1, len(history_lstm.history['loss']) + 1)
    epochs_trans = range(1, len(history_t.history['loss']) + 1)
    
    # Loss curves
    axes[0].plot(epochs_lstm, history_lstm.history['loss'], 'o-', color='#00ccff', lw=2, label='LSTM Train')
    axes[0].plot(epochs_lstm, history_lstm.history['val_loss'], 's--', color='#00ccff', lw=2, alpha=0.5, label='LSTM Val')
    axes[0].plot(epochs_trans, history_t.history['loss'], 'o-', color='#ff6600', lw=2, label='Transformer Train')
    axes[0].plot(epochs_trans, history_t.history['val_loss'], 's--', color='#ff6600', lw=2, alpha=0.5, label='Transformer Val')
    axes[0].set_xlabel('Epoch', fontsize=11, color='white'); axes[0].set_ylabel('Loss (MSE)', fontsize=11, color='white')
    axes[0].set_title('Training Loss Comparison', fontsize=12, fontweight='bold', color='white')
    axes[0].set_yscale('log')
    axes[0].legend(); axes[0].grid(True, alpha=0.2, color='#505050')
    axes[0].tick_params(colors='white')
    
    # MAE curves
    axes[1].plot(epochs_lstm, history_lstm.history['mae'], 'o-', color='#00ccff', lw=2, label='LSTM Train')
    axes[1].plot(epochs_lstm, history_lstm.history['val_mae'], 's--', color='#00ccff', lw=2, alpha=0.5, label='LSTM Val')
    axes[1].plot(epochs_trans, history_t.history['mae'], 'o-', color='#ff6600', lw=2, label='Transformer Train')
    axes[1].plot(epochs_trans, history_t.history['val_mae'], 's--', color='#ff6600', lw=2, alpha=0.5, label='Transformer Val')
    axes[1].set_xlabel('Epoch', fontsize=11, color='white'); axes[1].set_ylabel('MAE (SoC units)', fontsize=11, color='white')
    axes[1].set_title('Mean Absolute Error Progression', fontsize=12, fontweight='bold', color='white')
    axes[1].legend(); axes[1].grid(True, alpha=0.2, color='#505050')
    axes[1].tick_params(colors='white')
    
    plt.tight_layout()
    plt.show()